# Geopack SDK: Quotas, Generated Files, and Dataset APIs

Covers Phase 1b endpoints:
- `client.quotas.my_summary()` - fail fast before upload/workflow
- `client.generated_files` - list / download / delete
- `client.datasets` - `delete`, `query`, `discover`, ACL helpers

Requires a running Geopack API and credentials in `.env`.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from dotenv import load_dotenv

source_path = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if os.path.exists(source_path) and source_path not in sys.path:
    sys.path.insert(0, source_path)

from geopack_sdk import GeopackClient

load_dotenv()
client = GeopackClient(base_url=os.getenv("GEOPACK_API_URL", "http://localhost:3000/api"))
client.auth.login(
    os.getenv("GEOPACK_USERNAME", "admin"),
    os.getenv("GEOPACK_PASSWORD", "password"),
)
print("Logged in as", client.users.me().userName)

Logged in as admin


## 1. Quota summary (fail fast)

Check limits before starting a workflow or large upload.

In [2]:
summary = client.quotas.my_summary()
print(f"Quotas enabled: {summary.enabled} ({summary.reason})")
if summary.plan:
    print(f"Plan: {summary.plan.displayName} ({summary.plan.code})")

wf_key = "workflow.count.runs.daily"
if client.quotas.is_over_limit(wf_key, summary=summary):
    print(f"STOP: over limit for {wf_key}")
else:
    remaining = client.quotas.remaining_for(wf_key, summary=summary)
    print(f"{wf_key}: remaining={remaining}")

for row in client.quotas.warn_limits(summary=summary):
    pct = row.percentageUsed
    pct_str = f"{pct:.0f}" if pct is not None else "?"
    print(f"  WARN {row.dimensionKey}: {pct_str}% used")

Quotas enabled: True (ok)
Plan: Standard (STANDARD)
workflow.count.runs.daily: remaining=98.0


## 2. Generated files

List and optionally download outputs registered in **Generated Files** (exports, workflows).

In [3]:
files = client.generated_files.list(page_size=5)
print(f"Total generated files: {files.totalItems}")
for f in files.items:
    print(f"  [{f.id}] {f.fileName} ({f.fileSize} bytes)")

# Uncomment to download the first file:
# if files.items:
#     path = client.generated_files.download(files.items[0].id, "output/")
#     print("Downloaded:", path)

Total generated files: 95
  [135] Rural_District_shp.gpkg (2535424 bytes)
  [134] Rural_District_shp.gpkg (2535424 bytes)
  [133] Rural_District_shp.gpkg (2535424 bytes)
  [132] Rural_District_shp.gpkg (2535424 bytes)
  [131] Rural_District_shp.gpkg (2535424 bytes)


## 3. Dataset query (structured)

`POST /datasets/{id}/query` expects a **FeatureQuery DSL** (`pagination`, `projection`, optional `filter`) — not bare `limit`/`offset` at the top level. Use `build_simple_query()` or `query(..., limit=5)`.

In [4]:
from geopack_sdk.datasets import build_simple_query

ds_page = client.datasets.list(page_size=20, active_filters={"dataType": "vector"})
vector_ds = [d for d in ds_page.datasets if d.dataType == "vector"]

if not vector_ds:
    print("No vector datasets available for query demo.")
else:
    ds_id = vector_ds[0].id
    # Option A: explicit DSL
    fc = client.datasets.query(ds_id, build_simple_query(limit=5, offset=0))
    # Option B: shorthand — client.datasets.query(ds_id, limit=5)
    print(f"Dataset {ds_id} ({vector_ds[0].name}): {len(fc.features)} features returned")

Dataset 2408 (Rural_District.shp): 118 features returned


## 4. Dataset ACL (read-only demo)

Requires `dataset:share` or admin permission.

In [5]:
if ds_page.datasets:
    ds_id = ds_page.datasets[0].id
    try:
        acls = client.datasets.get_acls(ds_id)
        print(f"ACL entries for dataset {ds_id}: {len(acls)}")
        for acl in acls[:5]:
            label = acl.permissionName or acl.permissionId
            print(f"  {acl.principalType}#{acl.principalId} -> {label}")
    except Exception as e:
        print(f"ACL not available: {e}")

ACL entries for dataset 2408: 0


## 5. Dataset discover (after temp upload)

Use the same `sourceFiles` metadata as upload (`sessionId`, `relativePath`, `originalName`).

In [ ]:
# Example only - run after uploading to /uploads/temp in notebook 04
# discovery = client.datasets.discover(
#     source_files=[{
#         "sessionId": "...",
#         "relativePath": "file.geojson",
#         "originalName": "file.geojson",
#     }],
#     data_store_id=1,
#     workgroup_id=1,
#     wait=False,
# )
# if discovery.is_background_task:
#     print("Background task:", discovery.taskId)
# else:
#     print("Discovered:", discovery.count, "layers")